# Du tar en förtränad språkmodell (t.ex. **Flan-T5**) och tränar den specifikt på din typ av loggar.

Du säger alltså till modellen:

🗣️ “Hej, när du ser ett **loggmeddelande** som detta, ska du **omvandla** det till en mall som denna.”

Exempel:
Input:  "Error: Failed to connect to database at **192.168.1.1**"
Output: "Error: Failed to connect to database at **<IP>**"

Modellen lär sig att förstå **mönster i loggdata** — vad är **dynamiskt** (som **IP-adresser, användarnamn, sökvägar**) och vad är **statiskt (själva feltexten)**.

In [ ]:
# Importera nödvändiga bibliotek
from transformers import T5Tokenizer, T5ForConditionalGeneration  # För att använda T5-tokenizer och modell
from torch.utils.data import DataLoader, Dataset  # För att skapa dataset och dataladdare
import torch  # PyTorch-biblioteket

# Definiera en datasetklass för loggdata
class LogDataset(Dataset):
    def __init__(self, log_texts, templates):
        """
        Initialiserar datasetet med loggmeddelanden och deras motsvarande mallar.

        Args:
            log_texts (list): Lista med loggmeddelanden.
            templates (list): Lista med motsvarande mallar för loggmeddelandena.
        """
        self.log_texts = log_texts  # Spara loggmeddelanden
        self.templates = templates  # Spara mallar
        self.tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')  # Initialisera T5-tokenizer

    def __len__(self):
        """
        Returnerar antalet loggmeddelanden i datasetet.
        """
        return len(self.log_texts)

    def __getitem__(self, idx):
        """
        Hämtar ett loggmeddelande och dess mall vid en given index.

        Args:
            idx (int): Index för det loggmeddelande som ska hämtas.

        Returns:
            dict: En ordbok med 'input_ids', 'attention_mask' och 'labels'.
        """
        input_text = self.log_texts[idx]  # Hämta loggmeddelande vid index idx
        target_text = self.templates[idx]  # Hämta motsvarande mall
        # Tokenisera loggmeddelandet
        input_encoding = self.tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
        # Tokenisera mallen
        target_encoding = self.tokenizer(target_text, return_tensors="pt", padding=True, truncation=True)
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),  # Indata-ID för modellen
            'attention_mask': input_encoding['attention_mask'].squeeze(),  # Uppmärksamhetsmask för indata
            'labels': target_encoding['input_ids'].squeeze()  # Mål-ID för modellen
        }

# Exempel på loggmeddelanden och motsvarande mallar
log_texts = [
    "Error: Failed to connect to database at 192.168.1.1",  # Exempel på felmeddelande
    "Warning: Disk space running low on /dev/sda1",  # Exempel på varningsmeddelande
    "Error: Could not establish connection to host 172.16.0.1",
    "Info: User 'johndoe' logged in successfully",
    "Warning: System memory is critically low",
    "Error: File not found: /path/to/file",
    "Info: Service 'XYZ' started successfully",
    "Warning: High CPU usage on core 0",
    "Error: Permission denied for user 'admin'",
    "Info: Database connection established",
    "Debug: Entering function 'calculate_sum'",
    "Error: Invalid input parameters",
    "Warning: Network connection lost",
    "Info: User 'guest' logged out",
    "Error: Out of memory",
    "Critical: System failure detected",
    "Notice: Configuration file updated",
    "Alert: Intrusion attempt detected from 10.0.0.1",
    "Emergency: Power supply failure imminent",
    "Info: New user 'janeDoe' registered",
    "Warning: Temperature exceeding threshold",
    "Error: Transaction rollback failed",
    "Debug: Exiting function 'process_data'",
    "Info: Backup completed successfully",
    "Warning: Certificate expiring soon",
    "Error: Invalid license key",
    "Critical: Database corrupted",
    "Notice: Scheduled maintenance starting in 5 minutes",
    "Alert: High network latency detected",
    "Emergency: Disk drive failure detected",
    "Info: Application updated to version 2.0",
    "Warning: Battery level low",
    "Error: Failed to send email to user@example.com",
    "Debug: Variable 'x' value is 10",
    "Info: User 'johndoe' changed password",
    "Warning: Firewall rule blocked connection",
    "Error: SQL query syntax error",
    "Critical: System overload detected",
    "Notice: Server restarting",
    "Alert: Unauthorized access attempt",
    "Emergency: System shutting down"
]
templates = [
    "Error: Failed to connect to database at <IP>",  # Motsvarande mall med platshållare för IP
    "Warning: Disk space running low on <DEVICE>",  # Motsvarande mall med platshållare för enhet
    "Error: Could not establish connection to host <IP>",
    "Info: User '<USERNAME>' logged in successfully",
    "Warning: System memory is critically low",
    "Error: File not found: <FILEPATH>",
    "Info: Service '<SERVICENAME>' started successfully",
    "Warning: High CPU usage on core <CORENUMBER>",
    "Error: Permission denied for user '<USERNAME>'",
    "Info: Database connection established",
    "Debug: Entering function '<FUNCTIONNAME>'",
    "Error: Invalid input parameters",
    "Warning: Network connection lost",
    "Info: User '<USERNAME>' logged out",
    "Error: Out of memory",
    "Critical: System failure detected",
    "Notice: Configuration file updated",
    "Alert: Intrusion attempt detected from <IP>",
    "Emergency: Power supply failure imminent",
    "Info: New user '<USERNAME>' registered",
    "Warning: Temperature exceeding threshold",
    "Error: Transaction rollback failed",
    "Debug: Exiting function '<FUNCTIONNAME>'",
    "Info: Backup completed successfully",
    "Warning: Certificate expiring soon",
    "Error: Invalid license key",
    "Critical: Database corrupted",
    "Notice: Scheduled maintenance starting in <TIME>",
    "Alert: High network latency detected",
    "Emergency: Disk drive failure detected",
    "Info: Application updated to version <VERSION>",
    "Warning: Battery level low",
    "Error: Failed to send email to <EMAIL>",
    "Debug: Variable '<VARIABLENAME>' value is <VALUE>",
    "Info: User '<USERNAME>' changed password",
    "Warning: Firewall rule blocked connection",
    "Error: SQL query syntax error",
    "Critical: System overload detected",
    "Notice: Server restarting",
    "Alert: Unauthorized access attempt",
    "Emergency: System shutting down"
]

# Skapa dataset och dataladdare
dataset = LogDataset(log_texts, templates)  # Skapa en instans av LogDataset med loggmeddelanden och mallar
# Skapa en DataLoader med padding
dataloader = DataLoader(
    dataset,
    batch_size=2,  # Antal prover per batch
    shuffle=True,  # Om data ska slumpas
    collate_fn=lambda batch: {
        key: torch.nn.utils.rnn.pad_sequence(
            [d[key] for d in batch], batch_first=True, padding_value=0
        ) for key in batch[0]
    }  # Anpassad collate-funktion för att hantera padding
)

# Ladda förtränad Flan-T5-modell
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')  # Ladda den förtränade modellen

# Definiera träningsparametrar
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)  # AdamW-optimerare med inlärningshastighet 5e-5
num_epochs = 3  # Antal träningsiterationer över hela datasetet

# Finjusteringen äger rum i följande kodsegment
# Träningsloop
model.train()  # Sätt modellen i träningsläge
for epoch in range(num_epochs):
    for batch in dataloader:
        optimizer.zero_grad()  # Nollställ gradienterna
        input_ids = batch['input_ids']  # Hämta indata-ID från batchen
        attention_mask = batch['attention_mask']  # Hämta uppmärksamhetsmask från batchen
        labels = batch['labels']  # Hämta etiketter från batchen
        # Framåtpassering genom modellen
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss  # Beräkna förlusten
        loss.backward()  # Bakåtpassering för att beräkna gradienter
        optimizer.step()  # Uppdatera modellens vikter
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")  # Skriv ut förlusten för varje epok

# Spara den finjusterade modellen
model.save_pretrained('finetuned_flan_t5_logparser')  # Spara modellens vikter och konfiguration


Epoch 1, Loss: 1.2376629114151
Epoch 2, Loss: 0.12954175472259521
Epoch 3, Loss: 0.08446899056434631


Detaljerad förklaring av finjusteringsprocessen:
1. Aktivera träningsläge: model.train()
Aktiverar funktioner som är specifika för träning, såsom dropout och batchnormalisering.
2. Iterera över epoker: for batch in dataloader:
Loopar över de batcher av data som genereras av dataloader. Varje batch innehåller ett antal tränings-exempel som behandlas samtidigt för effektivitet.
3. Iterera över batcher: optimizer.zero_grad()
Loopar över de batcher av data som genereras av dataloader. Varje batch innehåller ett antal tränings-exempel som behandlas samtidigt för effektivitet.
4. Nollställ gradienterna: optimizer.zero_grad()
Rensar de ackumulerade gradienterna från föregående iteration för att förhindra att de blandas med gradienterna från den aktuella batchen.
5. Extrahera batchdata:
input_ids = batch['input_ids']
attention_mask = batch['attention_mask']
labels = batch['labels']
Hämtar input_ids, attention_mask och labels från den aktuella batchen.
6. Framåtpassering:  
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
Skickar indata genom modellen för att få prediktioner och beräkna förlusten mot de faktiska etiketterna.
7. Beräkna förlust: loss = outputs.loss
Extraherar förlustvärdet som indikerar hur långt modellens prediktioner är från de faktiska värdena.
8. Bakåtpassering: loss.backward()
Beräknar gradienterna för modellens parametrar baserat på förlusten, vilket är nödvändigt för att uppdatera vikterna korrekt.
9. Uppdatera vikter: optimizer.step()
Använder de beräknade gradienterna för att justera modellens vikter, vilket förbättrar dess prestanda på uppgiften.
10. Logga förlust: print(f"Epoch {epoch+1}, Loss: {loss.item()}")
Skriver ut förlusten efter varje epok för att övervaka träningsprocessen och modellens inlärning.

Förlustvärdet (loss) mäter skillnaden mellan modellens förutsägelser och de faktiska målvärdena. Målet är att minimera denna förlust under träningen. I detta fall minskar förlusten från epok 2 till epok 3, vilket indikerar att modellen börjar lära sig från data. Den initiala ökningen från epok 1 till epok 2 kan bero på flera faktorer, såsom inlärningshastighet, modellens initialisering eller datasetets komplexitet.

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Ladda den finjusterade modellen och tokenizern
model = T5ForConditionalGeneration.from_pretrained('finetuned_flan_t5_logparser')
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')

# Sätt modellen i utvärderingsläge
model.eval()

# Exempel på nya loggmeddelanden
new_logs = [
    "Error: Unable to reach server at 10.0.0.1",
    "Info: User admin logged in from 192.168.1.100"
]

# Bearbeta varje loggmeddelande
for log in new_logs:
    # Tokenisera indata
    inputs = tokenizer(log, return_tensors="pt", padding=True, truncation=True)
    # Generera utdata från modellen
    outputs = model.generate(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
    # Dekoda och skriv ut resultatet
    generated_template = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Original logg: {log}")
    print(f"Genererad mall: {generated_template}")
    print()


Original logg: Error: Unable to reach server at 10.0.0.1
Genererad mall: Error: Unable to reach server at 10.0.0.1

Original logg: Info: User admin logged in from 192.168.1.100
Genererad mall: Info: User admin logged in from 192.168.1.100

